# 8. N8N ORCHESTRATION PREPARATION
## Daily Customer Churn Predictor · VivaMarket Brasil

---

**INPUT:** `../data/processed/churn_predictions_YYYYMMDD.parquet`, `../data/processed/churn_explainability_YYYYMMDD.parquet`, and `../models/churn_scoring_package_YYYYMMDD.joblib`

**OUTPUT:** `../data/processed/retention_actions_YYYYMMDD.parquet`, `../n8n/daily_churn_retention_workflow_YYYYMMDD.json`, and `../reports/n8n_orchestration_YYYYMMDD.html`

*A production-minded retention-action payload and an n8n workflow blueprint aligned with the project retention strategy.*


---
## 8.1. STARTING SITUATION


The project now has risk scores, explainability outputs, and a deployment-ready scoring package. The remaining operational step is to define exactly what the daily orchestration should send downstream: who should be contacted, through which channels, with what incentive, and under which guardrails.

This notebook translates the confirmed retention strategy into an executable payload structure aligned with the canonical V2C policy and the professional action catalog defined for VivaMarket Brasil.

---
## 8.2. NOTEBOOK OBJECTIVE


- **Business objective:** convert scored customers into daily retention actions that match the High / Medium / Low framework from the confirmed retention strategy document.
- **Technical objective:** build a workflow-ready payload and a concrete n8n JSON blueprint that can later be implemented with minimal ambiguity.

In [1]:
import json
import logging
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import joblib
import pandas as pd
import numpy as np

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s', force=True)
logger = logging.getLogger('nb08_n8n_orchestration')
logger.info('NB08 started: n8n orchestration preparation.')


2026-05-26 20:46:29,715 | INFO | NB08 started: n8n orchestration preparation.


In [2]:
def resolve_required_artifact(directory: Path, prefix: str, run_date_tag: str, suffix: str) -> Path:
    candidate = directory / f'{prefix}_{run_date_tag}.{suffix}'
    if not candidate.exists():
        raise FileNotFoundError(f'Missing required artifact for run_date_tag={run_date_tag}: {candidate}')
    return candidate


def _artifact_key_set(path: Path):
    df = pd.read_parquet(path, columns=['customer_unique_id', 'snapshot_key'])
    return set(zip(df['customer_unique_id'].astype(str), df['snapshot_key'].astype(str)))


def select_consistent_scoring_bundle(models_dir: Path, processed_dir: Path):
    bundle_candidates = sorted(models_dir.glob('churn_scoring_package_*.joblib'), reverse=True)
    prediction_candidates = sorted(processed_dir.glob('churn_predictions_*.parquet'), reverse=True)
    explainability_candidates = sorted(processed_dir.glob('churn_explainability_*.parquet'), reverse=True)

    for candidate in bundle_candidates:
        bundle = joblib.load(candidate)
        metadata = bundle.get('metadata', {})
        run_date_tag = metadata.get('run_date_tag')
        if run_date_tag:
            prediction_candidate = processed_dir / f'churn_predictions_{run_date_tag}.parquet'
            explainability_candidate = processed_dir / f'churn_explainability_{run_date_tag}.parquet'
            if prediction_candidate.exists() and explainability_candidate.exists():
                return candidate, bundle, run_date_tag, prediction_candidate, explainability_candidate

        for prediction_candidate in prediction_candidates:
            for explainability_candidate in explainability_candidates:
                if _artifact_key_set(prediction_candidate) == _artifact_key_set(explainability_candidate):
                    resolved_tag = explainability_candidate.stem.split('_')[-1]
                    metadata.setdefault('resolved_prediction_file', prediction_candidate.name)
                    metadata.setdefault('resolved_explainability_file', explainability_candidate.name)
                    metadata.setdefault('run_date_tag', resolved_tag)
                    bundle['metadata'] = metadata
                    return candidate, bundle, resolved_tag, prediction_candidate, explainability_candidate

    raise FileNotFoundError('No scoring bundle matched a consistent prediction/explainability artifact pair, even after compatibility fallback.')


PROJECT_ROOT = Path.cwd().resolve().parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORTS_DIR = PROJECT_ROOT / 'reports'
N8N_DIR = PROJECT_ROOT / 'n8n'
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
N8N_DIR.mkdir(parents=True, exist_ok=True)

run_timestamp = datetime.now(ZoneInfo('Europe/Paris'))
run_datetime_label = run_timestamp.strftime('%Y-%m-%d %H:%M %Z')
scoring_bundle_path, scoring_bundle, run_date_tag, prediction_path, explainability_path = select_consistent_scoring_bundle(MODELS_DIR, PROCESSED_DIR)
run_id = scoring_bundle['metadata'].get('run_id', f'canonical_v2c_{run_date_tag}')
actions_path = PROCESSED_DIR / f'retention_actions_{run_date_tag}.parquet'
workflow_json_path = N8N_DIR / f'daily_churn_retention_workflow_{run_date_tag}.json'
error_workflow_json_path = N8N_DIR / f'error_handler_workflow_{run_date_tag}.json'
orchestration_html_path = REPORTS_DIR / f'n8n_orchestration_{run_date_tag}.html'
main_workflow_source_path = N8N_DIR / 'n8n_workflow_daily_churn_retention_workflow.json'
error_workflow_source_path = N8N_DIR / 'n8n_workflow_error_handler_workflow.json'
model_version = scoring_bundle['metadata'].get('model_version', scoring_bundle['metadata'].get('version_name', 'v2'))
pipeline_tag = scoring_bundle['metadata'].get('pipeline_tag', 'canonical_v2c_phase2')

logger.info('Orchestration notebook selected scoring bundle: %s', scoring_bundle_path)
logger.info('Orchestration notebook anchored to run_date_tag=%s', run_date_tag)
logger.info('Orchestration notebook anchored to run_id=%s', run_id)


2026-05-26 20:46:30,405 | INFO | Orchestration notebook selected scoring bundle: /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/models/churn_scoring_package_20260519.joblib


2026-05-26 20:46:30,406 | INFO | Orchestration notebook anchored to run_date_tag=20260519


2026-05-26 20:46:30,407 | INFO | Orchestration notebook anchored to run_id=canonical_v2c_20260519


In [3]:
prediction_df = pd.read_parquet(prediction_path)
explainability_df = pd.read_parquet(explainability_path)
action_df = prediction_df.merge(
    explainability_df[[
        'customer_unique_id', 'snapshot_key', 'top_driver_group', 'recommended_offer_type',
        'recommended_discount_pct', 'free_shipping_flag', 'vip_human_touch_flag', 'ltv_segment'
    ]],
    on=['customer_unique_id', 'snapshot_key'],
    how='left',
    validate='one_to_one',
)

action_df['top_driver_group'] = action_df['top_driver_group'].fillna('unassigned_sample_gap')
action_df['ltv_segment'] = action_df['ltv_segment'].astype(object).fillna('UNASSIGNED')

def normalize_offer(row):
    driver = row['top_driver_group']
    risk = row['risk_tier']
    if risk == 'HIGH':
        if driver == 'recency':
            return 'reactivacion_fuerte'
        if driver == 'frequency':
            return 'compra_recurrente'
        if driver == 'logistics':
            return 'desculpas_prioridade'
        if driver == 'monetary':
            return 'bundle_upsell_exclusivo'
        if driver == 'category':
            return 'cupon_categoria'
        return 'personalizado_categoria'
    if risk == 'MEDIUM':
        return 'nurturing_recomendaciones'
    return 'loyalty_fidelizacion'

action_df['recommended_offer_type'] = action_df.apply(normalize_offer, axis=1)
action_df['recommended_discount_pct'] = action_df['recommended_discount_pct'].fillna(
    action_df['risk_tier'].map({'HIGH': 25, 'MEDIUM': 12, 'LOW': 0})
)
action_df.loc[action_df['risk_tier'].eq('HIGH') & action_df['vip_human_touch_flag'].fillna(False), 'recommended_discount_pct'] = 30
action_df['free_shipping_flag'] = action_df['free_shipping_flag'].fillna(
    action_df['risk_tier'].map({'HIGH': True, 'MEDIUM': True, 'LOW': False})
)
action_df['vip_human_touch_flag'] = action_df['vip_human_touch_flag'].fillna(False)

action_df['primary_channels'] = action_df['risk_tier'].map({
    'HIGH': 'email,push',
    'MEDIUM': 'email,push',
    'LOW': 'email,in_app',
})
action_df['contact_policy'] = action_df['risk_tier'].map({
    'HIGH': 'day0_email_push__day7_escalated_email__day14_feedback_survey',
    'MEDIUM': 'every_3_to_7_days_nurturing',
    'LOW': 'weekly_or_monthly_loyalty_content',
})
action_df['message_focus'] = action_df.apply(
    lambda row: {
        ('HIGH', 'recency'): 'sentimos_sua_falta',
        ('HIGH', 'frequency'): 'volte_com_frequencia',
        ('HIGH', 'logistics'): 'envio_prioritario',
        ('HIGH', 'monetary'): 'kit_exclusivo_valor',
        ('HIGH', 'category'): 'volte_para_categoria',
        ('MEDIUM', 'unassigned_sample_gap'): 'recomendacoes_personalizadas',
    }.get((row['risk_tier'], row['top_driver_group']), 'fidelizacao_relacionamento'),
    axis=1,
)
# Phase 4 holdout baseline: deterministic customer-level assignment.
# We use a stable hash of customer_unique_id so assignment is reproducible across reruns
# and does not oscillate randomly within the same analytical window.
holdout_ratio = 0.15
holdout_window_days = 30
holdout_assignment_key = 'phase4_holdout_v1_customer_hash_mod_100'
holdout_hash = pd.util.hash_pandas_object(action_df['customer_unique_id'].astype(str), index=False).astype('uint64')
action_df['control_group_flag'] = ((holdout_hash % 100) < int(holdout_ratio * 100)).astype(bool)
action_df['holdout_reason'] = np.where(action_df['control_group_flag'], 'deterministic_phase4_holdout', None)
action_df['holdout_assignment_key'] = np.where(action_df['control_group_flag'], holdout_assignment_key, None)
action_df['holdout_window_days'] = np.where(action_df['control_group_flag'], holdout_window_days, None)
action_df['holdout_window_end'] = np.where(
    action_df['control_group_flag'],
    pd.to_datetime(action_df['snapshot_date']) + pd.to_timedelta(holdout_window_days, unit='D'),
    pd.NaT,
)
action_df['send_action_flag'] = ~action_df['control_group_flag']
action_df['offer_code_stub'] = action_df.apply(lambda row: f"{row['risk_tier'][:1]}-{row['snapshot_key']}-{str(row['customer_unique_id']).replace(' ', '').upper()}", axis=1)
action_df['journey_stage_count'] = action_df['risk_tier'].map({'HIGH': 4, 'MEDIUM': 2, 'LOW': 1})
action_df['run_id'] = run_id
action_df['run_date_tag'] = run_date_tag
action_df.to_parquet(actions_path, index=False)
logger.info('Retention actions parquet saved to %s', actions_path)
action_df.head()


2026-05-26 20:46:30,542 | INFO | Retention actions parquet saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/data/processed/retention_actions_20260519.parquet


,customer_unique_id,snapshot_key,snapshot_date,recency_days,total_orders,total_payment_value,orders_30d,orders_90d,observed_target,churn_probability,...,contact_policy,message_focus,control_group_flag,holdout_reason,holdout_assignment_key,holdout_window_days,holdout_window_end,send_action_flag,offer_code_stub,journey_stage_count
0,004288347e5e88a27ded2bb23747066c,20180401,2018-04-01,77,2,354.37,0.0,1.0,1,0.910062,...,weekly_or_monthly_loyalty_content,fidelizacao_relacionamento,False,NaN,NaN,None,NaT,True,L-20180401-004288347E5E88A27DED2BB23747066C,1
1,00cc12a6d8b578b8ebd21ea4e2ae8b27,20180401,2018-04-01,376,2,126.20,0.0,0.0,1,0.954483,...,weekly_or_monthly_loyalty_content,fidelizacao_relacionamento,False,NaN,NaN,None,NaT,True,L-20180401-00CC12A6D8B578B8EBD21EA4E2AE8B27,1
2,011b4adcd54683b480c4d841250a987f,20180401,2018-04-01,45,2,236.30,0.0,1.0,1,0.856885,...,weekly_or_monthly_loyalty_content,fidelizacao_relacionamento,False,NaN,NaN,None,NaT,True,L-20180401-011B4ADCD54683B480C4D841250A987F,1
3,013f4353d26bb05dc6652f1269458d8d,20180401,2018-04-01,124,2,356.39,0.0,0.0,1,0.646774,...,weekly_or_monthly_loyalty_content,fidelizacao_relacionamento,False,NaN,NaN,None,NaT,True,L-20180401-013F4353D26BB05DC6652F1269458D8D,1
4,015557c9912277312b9073947804a7ba,20180401,2018-04-01,335,2,315.12,0.0,0.0,1,0.967295,...,every_3_to_7_days_nurturing,fidelizacao_relacionamento,True,deterministic_phase4_holdout,phase4_holdout_v1_customer_hash_mod_100,30,2018-05-01,False,M-20180401-015557C9912277312B9073947804A7BA,2


In [4]:
workflow_definition = json.loads(main_workflow_source_path.read_text(encoding='utf-8'))
error_workflow_definition = json.loads(error_workflow_source_path.read_text(encoding='utf-8'))

workflow_json_path.write_text(json.dumps(workflow_definition, indent=2), encoding='utf-8')
error_workflow_json_path.write_text(json.dumps(error_workflow_definition, indent=2), encoding='utf-8')
logger.info('Main workflow JSON snapshot saved to %s', workflow_json_path)
logger.info('Error workflow JSON snapshot saved to %s', error_workflow_json_path)


2026-05-26 20:46:30,566 | INFO | Main workflow JSON snapshot saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/n8n/daily_churn_retention_workflow_20260519.json


2026-05-26 20:46:30,567 | INFO | Error workflow JSON snapshot saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/n8n/error_handler_workflow_20260519.json


In [5]:
orchestration_summary = (
    action_df.groupby(['risk_tier', 'recommended_offer_type', 'primary_channels'], observed=False)
    .agg(
        rows_n=('customer_unique_id', 'size'),
        send_action_rows=('send_action_flag', 'sum'),
        holdout_rows=('control_group_flag', 'sum'),
        avg_discount_pct=('recommended_discount_pct', 'mean'),
    )
    .reset_index()
    .sort_values(['risk_tier', 'rows_n'], ascending=[True, False])
)

workflow_node_summary = pd.DataFrame([
    {'workflow': workflow_definition.get('name', 'main'), 'nodes_n': len(workflow_definition.get('nodes', [])), 'active': workflow_definition.get('active', False)},
    {'workflow': error_workflow_definition.get('name', 'error handler'), 'nodes_n': len(error_workflow_definition.get('nodes', [])), 'active': error_workflow_definition.get('active', False)},
])

css = """
<style>
:root {
  --vm-primary: #005090;
  --vm-accent: #f39c12;
  --vm-high: #c0392b;
  --vm-medium: #8A6A00;
  --vm-low: #2e8b57;
  --vm-text: #1f2933;
  --vm-muted: #4A5568;
  --vm-border: #d9e2ec;
  --vm-bg: #f7fafc;
  --vm-card: #ffffff;
}
body {font-family: Arial, sans-serif; margin: 32px; color: var(--vm-text); background: var(--vm-bg);}
h1, h2 {color: var(--vm-primary);}
.section {background: var(--vm-card); border: 1px solid var(--vm-border); border-radius: 14px; padding: 22px; margin: 18px 0; box-shadow: 0 8px 24px rgba(15, 23, 42, 0.04);}
.grid {display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 20px; margin: 20px 0 28px;}
.card {background: var(--vm-card); border: 1px solid var(--vm-border); border-radius: 14px; padding: 18px 20px; box-shadow: 0 8px 24px rgba(15, 23, 42, 0.06);}
.card h3 {margin: 0 0 10px; font-size: 0.95rem; color: var(--vm-muted); text-transform: uppercase; letter-spacing: 0.04em;}
.card .value {font-size: 2rem; font-weight: 700; color: var(--vm-text);}
pre {white-space: pre-wrap; word-break: break-word; background: #0f172a; color: #e2e8f0; padding: 16px; border-radius: 12px; overflow-x: auto;}
table {border-collapse: collapse; width: 100%; font-size: 0.95rem;}
th, td {border: 1px solid var(--vm-border); padding: 10px; text-align: left;}
th {background: #edf2f7;}
.footer {margin-top: 28px; color: var(--vm-muted); font-size: 0.9rem; border-top: 1px solid var(--vm-border); padding-top: 16px;}
</style>
"""

cards_html = ''.join([
    f"<div class='card'><h3>{row['workflow']}</h3><div class='value'>{int(row['nodes_n'])}</div><div class='note'>Nodes in workflow · active={row['active']}</div></div>"
    for _, row in workflow_node_summary.iterrows()
])

html_parts = [
    '<html><head><meta charset="utf-8"><title>N8N Orchestration</title></head><body>',
    css,
    '<h1>N8N ORCHESTRATION BLUEPRINT</h1>',
    "<div class='grid'>" + cards_html + '</div>',
    '<div class="section"><h2>Workflow summary</h2>' + workflow_node_summary.to_html(index=False) + '</div>',
    '<div class="section"><h2>Main workflow definition</h2>' + f'<pre>{json.dumps(workflow_definition, indent=2)}</pre>' + '</div>',
    '<div class="section"><h2>Error workflow definition</h2>' + f'<pre>{json.dumps(error_workflow_definition, indent=2)}</pre>' + '</div>',
    '<div class="section"><h2>Action summary</h2>' + orchestration_summary.to_html(index=False) + '</div>',
    '<div class="section"><h2>Sample payload</h2>' + action_df.head(25).to_html(index=False) + '</div>',
    '<div class="section"><h2>Notes</h2><ul>'
    '<li>The HTML and dated JSON snapshots are aligned with the current n8n workflow files stored under <code>n8n/</code>.</li>'
    '<li>The main workflow now follows the two-workflow architecture: production pipeline + dedicated error handler.</li>'
    '<li>The retention payload is now normalized to the current operational reality: HIGH-tier primary channels are email and push until SMS is explicitly implemented in a later phase.</li><li>Phase 4 holdout baseline uses a deterministic customer-level assignment with a 30-day measurement window and explicit holdout metadata fields.</li>'
    '</ul></div>',
    f"<div class='footer'><strong>Run date:</strong> {run_datetime_label} &nbsp;|&nbsp; <strong>Model version:</strong> {model_version} &nbsp;|&nbsp; <strong>Pipeline tag:</strong> {pipeline_tag}</div>",
    '</body></html>'
]
orchestration_html_path.write_text('\n'.join(html_parts), encoding='utf-8')
logger.info('Orchestration report saved to %s', orchestration_html_path)
orchestration_summary.head(12)


2026-05-26 20:46:30,603 | INFO | Orchestration report saved to /data/.openclaw/workspace/projects/TFM/daily-customer-churn-predictor/reports/n8n_orchestration_20260519.html


,risk_tier,recommended_offer_type,primary_channels,rows_n,send_action_rows,holdout_rows,avg_discount_pct
1,HIGH,compra_recurrente,"email,push",530,449,81,25.716981
2,HIGH,reactivacion_fuerte,"email,push",97,79,18,25.206186
0,HIGH,bundle_upsell_exclusivo,"email,push",43,35,8,25.581395
3,LOW,loyalty_fidelizacion,"email,in_app",1673,1433,240,0.000000
4,MEDIUM,nurturing_recomendaciones,"email,push",1003,848,155,12.000000


---
## 8.3. NOTEBOOK CLOSURE


The orchestration stage now has a concrete action payload, an explicit control-group policy, and a daily n8n workflow blueprint aligned with the confirmed retention strategy. That means the project can move from model outputs to campaign operations without reinterpreting business rules every day.

The final notebook should consolidate these artifacts into a reporting view that helps monitor quality, campaign mix, and future model drift.